# RegMap embedder — retraining on the expanded catalog (v2)

**Run date:** see `training_report.json` · **Base:** `C:\Users\User\regmap-platform\models\regmap-embedder` ·
**Output:** `..\models\regmap-embedder-v2`

The released model `regmap-embedder@0.1` was fine-tuned on **159 pairs from a single framework pair**
(NIST 800-53 → HIPAA). After importing NIST's public-domain OSCAL catalogues and the CSF 2.0 OLIR
informative-reference export, the training data grew to **1,253+
pairs across 5 framework pairs**.

The paper's central finding is that *in-domain paired data is the primary driver* of mapping quality —
so this run tests that claim directly by holding the method constant and multiplying the data.

> **Every number in this notebook is read from the artifacts on disk** (`training_report.json`, the eval
> JSONs). Nothing is hand-transcribed.

## 1. Data provenance and the split protocols

Training pairs by mapping:

| mapping                    | train pairs |
|----------------------------|-------------|
| nist_80053__hipaa_security | 150         |
| nist_80171__nist_80053     | 118         |
| nist_csf2__nist_80053      | 483         |
| nist_csf2__nist_80171      | 228         |
| nist_csf2__nist_ssdf       | 24          |

Two protocols were built, because one number cannot answer both questions:

- **`query-holdout`** — holds out *source* controls; the target corpus stays fully visible. This is how
  retrieval works in the product. **Used to decide whether the product improved.**
- **`strict`** — assigns every control id to one split *globally*; a pair is kept only if both endpoints
  agree. Needed because mappings compose through hubs, so the same control appears in several mappings in
  different roles (`AC-2` is a *target* in CSF→800-53 and a *source* in 800-53→HIPAA). Holding it out of
  only one still leaves the model with an optimised representation of its exact text.

Both splits are leakage-checked in code and reported CLEAN. Seed 42, matching the released
model so the figures stay comparable.

In [ ]:
!python regmap_engine/build_training_data.py --write

## 2. Baseline — what `@0.1` scores on genuinely held-out data

`eval_crosswalk.py` has **no train/test split**, so on the pair the model was fine-tuned on it reports
R@5 ≈ 0.91 and means nothing. `eval_retrain.py` ranks each held-out query against the **full** target
framework corpus, which is the realistic setting.

In [2]:
!python regmap_engine/eval_retrain.py --model models/regmap-embedder \
    --protocol query-holdout --compare-to tfidf --json-out data/processed/retrain-v2/eval_baseline.json

{
  "protocol": "query-holdout",
  "split": "test",
  "queries": 64,
  "results": {
    "TF-IDF": {
      "pooled_r@5": 0.4701832022144522,
      "ci": [
        0.3667376893939394,
        0.578125
      ],
      "n": 64,
      "pooled": {
        "recall@1": 0.2984,
        "hit@1": 0.5,
        "ndcg@1": 0.5,
        "recall@3": 0.4247,
        "hit@3": 0.6094,
        "ndcg@3": 0.4681,
        "recall@5": 0.4702,
        "hit@5": 0.6719,
        "ndcg@5": 0.4771,
        "recall@10": 0.5436,
        "hit@10": 0.7344,
        "ndcg@10": 0.4984,
        "mrr": 0.5913,
        "map": 0.4719
      }
    },
    "../models/regmap-embedder": {
      "pooled_r@5": 0.6532506555944055,
      "ci": [
        0.5545199592074592,
        0.751467803030303
      ],
      "n": 64,
      "pooled": {
        "recall@1": 0.3249,
        "hit@1": 0.5938,
        "ndcg@1": 0.5938,
        "recall@3": 0.5712,
        "hit@3": 0.75,
        "ndcg@3": 0.5973,
        "recall@5": 0.6533,
        "hit@5": 

## 3. Training

`MultipleNegativesRankingLoss` is implemented explicitly in `train_embedder.py` rather than called from
the library, because **neither sentence-transformers training API works in this environment**: the v3+
`SentenceTransformerTrainer` requires `datasets`, and the legacy `model.fit()` in 5.6.1 is only a shim
that also requires it — while `datasets` is unusable here because pyarrow's `_dataset` DLL is blocked by
an Application Control policy. (Installing it additionally breaks `import sentence_transformers`.)

The objective is the standard one: score every anchor against every positive in the batch, scale by
20.0, treat the diagonal as correct via cross-entropy. The model's pipeline ends in `Normalize`, so a
dot product *is* cosine similarity.

| setting         | value                                                                |
|-----------------|----------------------------------------------------------------------|
| loss            | MultipleNegativesRankingLoss (explicit in-batch softmax, scale 20.0) |
| seed            | 42                                                                   |
| epochs          | 10                                                                   |
| batch_size      | 16                                                                   |
| lr              | 2e-05                                                                |
| weight_decay    | 0.01                                                                 |
| grad_clip       | 1.0                                                                  |
| warmup_steps    | 62                                                                   |
| steps_per_epoch | 62                                                                   |
| total_steps     | 620                                                                  |
| train_pairs     | 1003                                                                 |
| val_pairs       | 250                                                                  |
| hard_negatives  | False                                                                |
| embedding_dim   | 384                                                                  |
| max_seq_length  | 256                                                                  |
| train_seconds   | 4468.2                                                               |

In [ ]:
!python regmap_engine/train_embedder.py --out models/regmap-embedder-v2 --protocol query-holdout

### Loss curve (held-out validation loss per epoch)

| epoch | train loss | val loss (held out) |
|-------|------------|---------------------|
| 0     | —          | 3.2646              |
| 1     | 2.1406     | 2.6161              |
| 2     | 1.5007     | 2.536               |
| 3     | 1.2871     | 2.6387              |
| 4     | 1.1317     | 2.6236              |
| 5     | 1.061      | 2.8399              |
| 6     | 1.0141     | 2.7419              |
| 7     | 0.992      | 2.7693              |
| 8     | 0.9878     | 2.7108              |
| 9     | 0.9193     | 2.6991              |
| 10    | 0.8884     | 2.7019              |

Validation loss **3.2646 → 2.7019**.

## 4. Result — v2 on the same held-out queries

In [4]:
!python regmap_engine/eval_retrain.py --model models/regmap-embedder-v2 \
    --protocol query-holdout --compare-to tfidf --json-out data/processed/retrain-v2/eval_v2.json

{
  "protocol": "query-holdout",
  "split": "test",
  "queries": 64,
  "results": {
    "../models/regmap-embedder-v2": {
      "pooled_r@5": 0.7342821241258741,
      "ci": [
        0.647350305944056,
        0.8177556818181818
      ],
      "n": 64,
      "pooled": {
        "recall@1": 0.3101,
        "hit@1": 0.5312,
        "ndcg@1": 0.5312,
        "recall@3": 0.6736,
        "hit@3": 0.8594,
        "ndcg@3": 0.6611,
        "recall@5": 0.7343,
        "hit@5": 0.9062,
        "ndcg@5": 0.6709,
        "recall@10": 0.8239,
        "hit@10": 0.9531,
        "ndcg@10": 0.693,
        "mrr": 0.7005,
        "map": 0.6224
      }
    }
  }
}


### Pooled comparison, held-out queries

| metric    | @0.1  | v2    | delta  | relative |
|-----------|-------|-------|--------|----------|
| recall@1  | 0.325 | 0.310 | -0.015 | -4.6%    |
| recall@3  | 0.571 | 0.674 | +0.102 | +17.9%   |
| recall@5  | 0.653 | 0.734 | +0.081 | +12.4%   |
| recall@10 | 0.756 | 0.824 | +0.068 | +9.0%    |
| ndcg@10   | 0.640 | 0.693 | +0.053 | +8.3%    |
| mrr       | 0.695 | 0.701 | +0.005 | +0.8%    |
| map       | 0.569 | 0.622 | +0.053 | +9.3%    |

## 5. Model selection — validation loss is the wrong criterion here

The loss curve above bottoms early and then rises while training loss keeps falling, which reads as
overfitting. So a second run was trained for 4 epochs with checkpointing on **best
held-out loss** — it selected epoch 2 (val loss 2.5509).

Evaluating all three on the *same* held-out queries gives a result worth stating plainly:

| model            | val loss | R@1   | R@5   | nDCG@10 | R@5 95% CI  |
|------------------|----------|-------|-------|---------|-------------|
| @0.1 (released)  | 3.2646   | 0.325 | 0.653 | 0.640   | 0.555-0.751 |
| v2, epoch 10     | 2.7019   | 0.310 | 0.734 | 0.693   | 0.647-0.818 |
| v2-best, epoch 2 | 2.5509   | 0.334 | 0.682 | 0.664   | 0.591-0.774 |

**The model with the best validation loss has the worst Recall@5 of the two retrained models.**
Selecting on loss would have shipped the weaker model.

The reason is a mismatch of scale: MNRL's loss is computed against **15 in-batch candidates**, while the
product ranks against a **full framework corpus of up to ~1,000 controls**. Being confident among 15
neighbours is not the same skill as ranking one of a thousand. So the release candidate was selected on
held-out **Recall@5**, not on loss — `v2, epoch 10`.

Note also that epoch 2 has the best **Recall@1**. If the product needed a single answer rather than a
shortlist, the choice would flip. It does not: `/map` returns a ranked shortlist for an expert to confirm.

## 6. How to read this honestly

- The number to beat is the paper's **held-out multi-relevant R@5 = 0.797**, not the 0.906 that
  `eval_crosswalk.py` prints — that figure is inflated by training leakage and `/eval` self-reports
  `benchmark: false` for exactly this reason.
- The held-out test set is **small** (64 queries), so the
  bootstrap confidence intervals are wide. Do not over-read small differences.
- `models/regmap-embedder/` (`@0.1`) is **not** overwritten — it is the artifact the DTRAP submission
  describes, and overwriting it would make the paper's numbers unreproducible. v2 is a separate
  directory with its own `REGMAP_MODEL_VERSION`.
- The serving layer keys its embedding cache on model version + control text, so swapping models
  invalidates and re-warms the cache automatically; there is no stale-vector risk.